In [6]:
import pandas as pd
import sqlite3

In [2]:
connection = sqlite3.connect('../../data/checking-logs.sqlite')


In [8]:
query = '''
CREATE TABLE datamart AS
SELECT c.uid, c.labname, c.timestamp as first_commit_ts, MIN(p.datetime) AS first_view_ts FROM checker AS c
LEFT JOIN pageviews AS p ON c.uid = p.uid
    WHERE c.status = 'ready'
    AND c.numTrials = 1
    AND c.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
    AND c.uid LIKE 'user_%'
GROUP BY 1, 2
'''

In [9]:
connection.execute(query)
connection.commit()
df_datamart = pd.read_sql('SELECT * FROM datamart', connection)
df_datamart
#connection.execute('''DROP TABLE datamart''')

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
...,...,...,...,...
135,user_8,laba04s,2020-04-19 10:22:35.761944,None
136,user_8,laba05,2020-05-02 13:28:07.705193,None
137,user_8,laba06,2020-05-16 17:56:15.755553,None
138,user_8,laba06s,2020-05-16 20:01:07.900727,None


In [10]:
df_datamart['first_commit_ts'] = pd.to_datetime(df_datamart['first_commit_ts'], errors='coerce')
df_datamart['first_view_ts'] = pd.to_datetime(df_datamart['first_view_ts'], errors='coerce')
print(df_datamart.dtypes)

uid                        object
labname                    object
first_commit_ts    datetime64[ns]
first_view_ts      datetime64[ns]
dtype: object


In [11]:
test = df_datamart[df_datamart['first_view_ts'].notna()]
control = df_datamart[df_datamart['first_view_ts'].isna()]
print('test db:\n', test)
print('control db:\n',control)

test db:
          uid   labname            first_commit_ts              first_view_ts
0     user_1    laba04 2020-04-26 17:06:18.462708 2020-04-26 21:53:59.624136
1     user_1   laba04s 2020-04-26 17:12:11.843671 2020-04-26 21:53:59.624136
2     user_1    laba05 2020-05-02 19:15:18.540185 2020-04-26 21:53:59.624136
3     user_1    laba06 2020-05-17 16:26:35.268534 2020-04-26 21:53:59.624136
4     user_1   laba06s 2020-05-20 12:23:37.289724 2020-04-26 21:53:59.624136
5     user_1  project1 2020-05-14 20:56:08.898880 2020-04-26 21:53:59.624136
6    user_10    laba04 2020-04-25 08:24:52.696624 2020-04-18 12:19:50.182714
7    user_10   laba04s 2020-04-25 08:37:54.604222 2020-04-18 12:19:50.182714
8    user_10    laba05 2020-05-01 19:27:26.063245 2020-04-18 12:19:50.182714
9    user_10    laba06 2020-05-19 11:39:28.885637 2020-04-18 12:19:50.182714
10   user_10   laba06s 2020-05-20 07:37:31.175817 2020-04-18 12:19:50.182714
11   user_10  project1 2020-05-12 20:12:28.056618 2020-04-18 12:19

In [12]:
mean_first_view_ts = test['first_view_ts'].mean()
control.loc[:, 'first_view_ts'] = mean_first_view_ts
print('test db:\n', test)
print('control new db:\n', control)

test db:
          uid   labname            first_commit_ts              first_view_ts
0     user_1    laba04 2020-04-26 17:06:18.462708 2020-04-26 21:53:59.624136
1     user_1   laba04s 2020-04-26 17:12:11.843671 2020-04-26 21:53:59.624136
2     user_1    laba05 2020-05-02 19:15:18.540185 2020-04-26 21:53:59.624136
3     user_1    laba06 2020-05-17 16:26:35.268534 2020-04-26 21:53:59.624136
4     user_1   laba06s 2020-05-20 12:23:37.289724 2020-04-26 21:53:59.624136
5     user_1  project1 2020-05-14 20:56:08.898880 2020-04-26 21:53:59.624136
6    user_10    laba04 2020-04-25 08:24:52.696624 2020-04-18 12:19:50.182714
7    user_10   laba04s 2020-04-25 08:37:54.604222 2020-04-18 12:19:50.182714
8    user_10    laba05 2020-05-01 19:27:26.063245 2020-04-18 12:19:50.182714
9    user_10    laba06 2020-05-19 11:39:28.885637 2020-04-18 12:19:50.182714
10   user_10   laba06s 2020-05-20 07:37:31.175817 2020-04-18 12:19:50.182714
11   user_10  project1 2020-05-12 20:12:28.056618 2020-04-18 12:19

In [13]:
test.to_sql("test", connection, if_exists="replace", index=False)
control.to_sql("control", connection, if_exists="replace", index=False)
connection.close()